In [45]:
import os
import openai

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


### Downloading eval dataset from langsmith

In [50]:
client = Client()


In [51]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)


In [52]:
dataset


Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('17f4534e-093d-4dc7-8c3a-54f0ef4eb105'), created_at=datetime.datetime(2026, 5, 14, 15, 17, 23, 476976, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 14, 15, 17, 23, 476976, tzinfo=TzInfo(0)), example_count=21, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.3-arm64-arm-64bit', 'sdk_version': '0.8.4', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [53]:
list(client.list_examples(dataset_id=dataset.id, limit=10))


[<class 'langsmith.schemas.Example'>(id=31b66dd5-5030-4b1c-b0dd-08a3cf23b8e3, dataset_id=17f4534e-093d-4dc7-8c3a-54f0ef4eb105, link='https://smith.langchain.com/o/a60396b7-26e7-43d4-af3b-1ddaf09b780e/datasets/17f4534e-093d-4dc7-8c3a-54f0ef4eb105/e/31b66dd5-5030-4b1c-b0dd-08a3cf23b8e3'),
 <class 'langsmith.schemas.Example'>(id=b3c5adab-a0a6-45f1-af82-fa2eba7f0e5a, dataset_id=17f4534e-093d-4dc7-8c3a-54f0ef4eb105, link='https://smith.langchain.com/o/a60396b7-26e7-43d4-af3b-1ddaf09b780e/datasets/17f4534e-093d-4dc7-8c3a-54f0ef4eb105/e/b3c5adab-a0a6-45f1-af82-fa2eba7f0e5a'),
 <class 'langsmith.schemas.Example'>(id=44aa279a-a401-44bb-9255-3031b46d772a, dataset_id=17f4534e-093d-4dc7-8c3a-54f0ef4eb105, link='https://smith.langchain.com/o/a60396b7-26e7-43d4-af3b-1ddaf09b780e/datasets/17f4534e-093d-4dc7-8c3a-54f0ef4eb105/e/44aa279a-a401-44bb-9255-3031b46d772a'),
 <class 'langsmith.schemas.Example'>(id=4a393c32-bdd6-4fc4-9fc7-1e21a74dd835, dataset_id=17f4534e-093d-4dc7-8c3a-54f0ef4eb105, link='htt

In [54]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs


{'ground_truth': "I'm sorry, but I currently do not have the information regarding the latest beauty trends.",
 'reference_context_ids': [],
 'reference_descriptions': []}

In [55]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs


{'question': 'What are the latest beauty trends for this season?'}

In [56]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[5].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[5].outputs


### Combining RAG pipeline

In [60]:
from dotenv import load_dotenv
from groq import Groq
from qdrant_client import QdrantClient
from openai import OpenAI
import os


load_dotenv("../.env", override=True)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
      api_key=os.environ.get("OPENROUTER_API_KEY"),

)
qdrant_client = QdrantClient(host="localhost", port=6333)

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)






In [61]:

import logging

logger = logging.getLogger(__name__)


def get_embedding(text: str, model="openai/text-embedding-3-small"):
    response = openrouter_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding














In [62]:
def retrieve_products(query: str, limit: int = 3):
    """
    Search for products based on a natural language query.
    """
    # 1. Convert the user's text query into a vector embedding
    query_vector = get_embedding(query)
    
    # 2. Search the Qdrant database for the closest matching vectors
    search_results = qdrant_client.query_points(
        collection_name="products",
        query=query_vector,
        limit=limit
    )


    retrieve_context = []
    average_rating = []
    similarity_score = []
    context_id = []

    for result in search_results.points:
        # Extract fields from the payload and the Qdrant result object
        retrieve_context.append(result.payload["description"])
        average_rating.append(result.payload["average_rating"])
        similarity_score.append(result.score)
        context_id.append(result.payload["parent_asin"])
        
    return {
        "retrieved_context": retrieve_context,
        "retrieved_context_ratings": average_rating,
        "similarity_score": similarity_score,
        "retrieved_context_ids": context_id
    }


In [63]:

def process_context(context):
    formatted_context = ""
    
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        
    return formatted_context

In [64]:

def build_prompt(preprocessed_context, question):
        prompt = f"""
        You are a shopping assistant that can answer questions about the products in stock.

        You will be given a question and a list of context.

        Instructions:
        - You need to answer the question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
        """
        return prompt

In [65]:
def generate_answer(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )

    return response.choices[0].message.content


In [66]:
def rag_pipeline(question: str, top_k: int = 5) -> str:
    try:
        retrieved_context = retrieve_products(question, limit=top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_prompt(preprocessed_context, question)
        answer = generate_answer(prompt)


        final_result={
            "answer":answer,
            "question":question,
            "retrieved_context":retrieved_context["retrieved_context"],
            "average_rating":retrieved_context["retrieved_context_ratings"],
            "similarity_score": retrieved_context["similarity_score"],
            "context_id":retrieved_context["retrieved_context_ids"]
        }


        return final_result
    except Exception as e:
        logger.error(f"Error in RAG pipeline: {e}")
        return {
            "answer": "Sorry, there was an error processing your request.",
            "question": question,
            "retrieved_context": [],
            "average_rating": [],
            "similarity_score": [],
            "context_id": []
        }

In [67]:
rag_pipeline("I need something to moisturize my skin, what do you recommend?")

{'answer': 'Based on the available products, I would recommend the following products for moisturizing skin:\n\n1. ID: B07S2HKRVM, rating: 3.9 - This product is described as lightweight, paraben-free, intensely hydrating, and provides an antioxidant defense against environmental pollution.\n2. ID: B00F97Y8RA, rating: 4.5 - This product is a ‘Island Spice’ Magnesium Cream that has a light texture and scent and provides a silky smooth finish. \n3. ID: B01IA954Q2, rating: 4.3, - Aloe Vera helps to leave your skin sleeker, smoother and softer.\n4. ID: B01G09YUSC, rating: 4.6, - Black pearl & gold hydrogel eye patch can help moisturize the skin.\n5. ID: B000PY8B98, rating: 4.4, - Although described as an aftershave balm, the description does also mention hydrating and moisturizing properties, making it suitable for moisturizing skin.\n\nHowever, if I had to recommend one, I would recommend ID: B07S2HKRVM. It is specifically designed for hydration and has a wide range of benefits such as red

#### RAGAS Metrics

In [68]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy


/var/folders/fl/mzsw4knd0v319n6f93hr8j7m0000gn/T/ipykernel_51350/581455323.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/fl/mzsw4knd0v319n6f93hr8j7m0000gn/T/ipykernel_51350/581455323.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/fl/mzsw4knd0v319n6f93hr8j7m0000gn/T/ipykernel_51350/581455323.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecat

In [ ]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings


ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))



/var/folders/fl/mzsw4knd0v319n6f93hr8j7m0000gn/T/ipykernel_51350/1358832598.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
/var/folders/fl/mzsw4knd0v319n6f93hr8j7m0000gn/T/ipykernel_51350/1358832598.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [70]:
result = rag_pipeline(reference_input["question"])

In [71]:
result

{'answer': 'Based on the available products, you can use the wig cap that comes with the products (ID: B07QJFVNF7 and ID: B0794MXH1S) which is designed to fix the wig and keep the hairstyle in place. Additionally, the Staytight Barrettes from ID: B00BUS8TTK can also be used to secure your hairstyle and keep it intact.',
 'question': 'What accessories can I use to keep my hairstyle intact?',
 'retrieved_context': ['About Our Wig 1.Rose net cap, more breathable and ajustable. 2.Our wig was made from premium synthetic fiber, please wash it with the shampoo the first time wearing. 3.All synthetic hair wigs are heat safe and can be curled and flat ironed. Please note that the heat settings must below 150c/302f. 4.All our wigs are with adjustable inner cap net, one size fits most. 5.You can cut or trim the wig into any style you like. Specification Type of hair: premium heat resistant synthetic fiber.  Color: Rainbow.  Weight:230g(± 20g)/set.  Length:35cm/14"(±5cm/2"). Warm Tips 1.The color 

In [72]:
async def ragas_faithfulness(run, example):
    
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)


In [73]:
await ragas_faithfulness(result, "")


0.6

In [74]:
async def ragas_responce_relevancy(run, example):
    
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    
    return await scorer.single_turn_ascore(sample)


In [75]:
await ragas_responce_relevancy(result, "")


np.float64(0.7144049048785486)

In [80]:
async def ragas_context_precision_id_based(run, example):
    
    sample = SingleTurnSample(
        retrieved_context_ids=run["context_id"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextPrecision()
    
    return await scorer.single_turn_ascore(sample)


In [ ]:
await ragas_context_precision_id_based(result, reference_output)

0.2

In [82]:
async def ragas_context_recall_id_based(run, example):
    
    sample = SingleTurnSample(
        retrieved_context_ids=run["context_id"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextRecall()
    
    return await scorer.single_turn_ascore(sample)


In [ ]:
await ragas_context_recall_id_based(result, reference_output)

0.5